In [1]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

import torch.nn as nn
from torchvision import models
import torch.optim as optim

import os
import torch
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
import numpy as np


# GPU & performance settings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Enable cuDNN autotuner for faster convs
torch.backends.cudnn.benchmark = True

# Allow TensorFloat-32 (speeds up matmul/conv on Ampere+ GPUs, safe for training)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# PyTorch 2.x precision control (A100/H100 benefit most)
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision('high')  # 'medium' is safer if unstable

# Check if BF16 is supported, otherwise fallback to FP16
bf16_ok = torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if bf16_ok else torch.float16

In [2]:

# Transformations 
transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Dataset 
data_dir = "DIR/horse/sapi279d-test_workspace/train"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

print(f"Total images: {len(dataset)}")
print(f"Number of classes: {len(dataset.classes)}")
print("Class names (subfolder labels):", dataset.classes[:10], "...")

# Stratified Split
targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []

for class_id in np.unique(targets):
    idxs = np.where(targets == class_id)[0]
    # first split train vs temp (val+test)
    train_ids, temp_ids = train_test_split(
        idxs, test_size=0.15, random_state=42, shuffle=True
    )
    # then split temp into val vs test
    val_ids, test_ids = train_test_split(
        temp_ids, test_size=1/3, random_state=42, shuffle=True
    )  # 1/3 of 15% = 5%
    
    train_idx.extend(train_ids)
    val_idx.extend(val_ids)
    test_idx.extend(test_ids)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# Subsets 
train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

# DataLoaders
batch_size = 128
num_workers = min(8, os.cpu_count())

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)

print("DataLoaders ready")

Total images: 100000
Number of classes: 200
Class names (subfolder labels): ['n01443537', 'n01629819', 'n01641577', 'n01644900', 'n01698640', 'n01742172', 'n01768244', 'n01770393', 'n01774384', 'n01774750'] ...
Train: 85000, Val: 10000, Test: 5000
DataLoaders ready


In [3]:
from torchvision.models import shufflenet_v2_x1_0, ShuffleNet_V2_X1_0_Weights

# Device 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Model 
# Use new weights API (replaces pretrained=True)
weights = ShuffleNet_V2_X1_0_Weights.DEFAULT
model = shufflenet_v2_x1_0(weights=weights)

# Adjust final layer for 200 classes
num_classes = 200
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Move to GPU with channels_last memory format (faster convolutions)
model = model.to(device, memory_format=torch.channels_last)

# Optionally compile (PyTorch 2.x only)
if hasattr(torch, "compile"):
    model = torch.compile(model, mode="max-autotune", fullgraph=False, dynamic=False)

# Loss function 
criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)

# Optimizer
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9,
                      weight_decay=1e-4, nesterov=True)


# AMP setup 
bf16_ok = torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if bf16_ok else torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=not bf16_ok)

Using device: cuda


In [4]:

# Training & Evaluation Functions (optimized for GPU)


def evaluate_model(model, loader, device):
    """Evaluate model accuracy on a given DataLoader"""
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total



def train_model(model, train_loader, val_loader, criterion, optimizer, device,
                amp_dtype, scaler, epochs=5):
    """Train the model and validate after each epoch"""
    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # forward pass with AMP
            with torch.autocast("cuda", dtype=amp_dtype):
                outputs = model(images)
                loss = criterion(outputs, labels)

            # backward pass with scaler
            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = 100 * correct / total
        val_acc = evaluate_model(model, val_loader, device)
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {running_loss/total:.4f} | "
              f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")


In [5]:
from tqdm import tqdm

def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=5):
    scaler = torch.cuda.amp.GradScaler()   # auto handles fp16/bf16
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

        for images, labels in progress_bar:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast("cuda", dtype=amp_dtype):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            progress_bar.set_postfix(loss=loss.item())

        # Validation (kept simple, no AMP)
        val_acc = evaluate_model(model, val_loader, device)
        train_acc = 100 * correct / total
        print(f"Epoch [{epoch+1}/{epochs}] "
              f"| Train Loss: {running_loss/total:.4f} "
              f"| Train Acc: {train_acc:.2f}% "
              f"| Val Acc: {val_acc:.2f}%")


In [6]:
epochs = 20
train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs)


Epoch 1/20:   0%|          | 0/665 [00:00<?, ?it/s]AUTOTUNE convolution(128x3x224x224, 24x3x3x3)
  triton_convolution_4 0.4680 ms 100.0%
  triton_convolution_3 0.5161 ms 90.7%
  convolution 0.5222 ms 89.6%
  triton_convolution_2 0.6328 ms 73.9%
  triton_convolution_0 0.6400 ms 73.1%
  triton_convolution_5 0.7188 ms 65.1%
  triton_convolution_1 1.0214 ms 45.8%
SingleProcess AUTOTUNE takes 13.7242 seconds
AUTOTUNE mm(100352x24, 24x58)
  triton_mm_6 0.0195 ms 100.0%
  triton_mm_7 0.0205 ms 95.0%
  triton_mm_8 0.0205 ms 95.0%
  triton_mm_13 0.0205 ms 95.0%
  triton_mm_9 0.0215 ms 90.5%
  triton_mm_10 0.0215 ms 90.5%
  triton_mm_14 0.0215 ms 90.5%
  triton_mm_17 0.0215 ms 90.5%
  triton_mm_12 0.0225 ms 86.4%
  triton_mm_11 0.0236 ms 82.6%
SingleProcess AUTOTUNE takes 3.5566 seconds
AUTOTUNE mm(401408x24, 24x58)
  triton_mm_18 0.0584 ms 100.0%
  triton_mm_20 0.0584 ms 100.0%
  triton_mm_21 0.0584 ms 100.0%
  triton_mm_26 0.0584 ms 100.0%
  triton_mm_29 0.0584 ms 100.0%
  triton_mm_19 0.0594 

Epoch [1/20] | Train Loss: 3.8166 | Train Acc: 26.12% | Val Acc: 35.82%


Epoch [2/20] | Train Loss: 2.6035 | Train Acc: 50.84% | Val Acc: 45.21%


Epoch [3/20] | Train Loss: 2.4040 | Train Acc: 56.46% | Val Acc: 46.98%


Epoch [4/20] | Train Loss: 2.2880 | Train Acc: 59.54% | Val Acc: 48.59%


...

Epoch [16/20] | Train Loss: 1.8476 | Train Acc: 72.54% | Val Acc: 47.62%


Epoch [17/20] | Train Loss: 1.8311 | Train Acc: 72.91% | Val Acc: 52.31%


Epoch [18/20] | Train Loss: 1.8152 | Train Acc: 73.45% | Val Acc: 44.52%


Epoch [19/20] | Train Loss: 1.8078 | Train Acc: 73.67% | Val Acc: 48.38%


Epoch [20/20] | Train Loss: 1.7939 | Train Acc: 74.01% | Val Acc: 50.97%


In [7]:
# Save only weights (state_dict)
torch.save(model.state_dict(), "shufflenetv2_weights.pth")

In [8]:
# Evaluate final model on test data
test_acc = evaluate_model(model, test_loader, device)
print(f"Final Test Accuracy: {test_acc:.2f}%")

AUTOTUNE convolution(8x3x224x224, 24x3x3x3)
  convolution 0.0430 ms 100.0%
  triton_convolution_1848 0.0635 ms 67.7%
  triton_convolution_1850 0.0655 ms 65.6%
  triton_convolution_1847 0.0666 ms 64.6%
  triton_convolution_1852 0.0681 ms 63.2%
  triton_convolution_1851 0.0748 ms 57.5%
  triton_convolution_1849 0.1290 ms 33.3%
SingleProcess AUTOTUNE takes 4.0249 seconds
AUTOTUNE mm(6272x24, 24x58)
  triton_mm_1853 0.0102 ms 100.0%
  triton_mm_1854 0.0102 ms 100.0%
  triton_mm_1859 0.0102 ms 100.0%
  triton_mm_1861 0.0102 ms 100.0%
  triton_mm_1862 0.0102 ms 100.0%
  triton_mm_1863 0.0102 ms 100.0%
  triton_mm_1864 0.0102 ms 100.0%
  mm 0.0113 ms 90.9%
  triton_mm_1855 0.0113 ms 90.9%
  triton_mm_1856 0.0113 ms 90.9%
SingleProcess AUTOTUNE takes 5.0751 seconds
AUTOTUNE mm(25088x24, 24x58)
  triton_mm_1866 0.0143 ms 100.0%
  triton_mm_1869 0.0143 ms 100.0%
  triton_mm_1871 0.0143 ms 100.0%
  triton_mm_1873 0.0143 ms 100.0%
  triton_mm_1865 0.0154 ms 93.3%
  triton_mm_1867 0.0154 ms 93.3%
 

Final Test Accuracy: 52.16%
